In [1]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
    .master("local[*]")
    .appName("NYC_Taxi_Feature_Engineering")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "localhost")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Setup complete!")

Spark version: 4.1.1
Setup complete!


In [2]:
DATA_PATH = r"C:\Users\MaK Tech\Documents\GitHub\nyc-taxi-bigdata\data\*.parquet"
df = spark.read.parquet(DATA_PATH)
print(f"Loaded {df.count():,} records, {len(df.columns)} columns")
df.printSchema()

Loaded 41,169,720 records, 19 columns
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [3]:
print("STEP 1: DATA CLEANING")
print("=" * 60)
print(f"Records before cleaning: {df.count():,}")

# Remove invalid records
df_clean = df.filter(
    (F.col("fare_amount") > 0) &
    (F.col("fare_amount") < 500) &
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") < 200) &
    (F.col("total_amount") > 0) &
    (F.col("total_amount") < 1000) &
    (F.col("passenger_count") > 0) &
    (F.col("passenger_count") <= 6) &
    (F.col("tpep_pickup_datetime").isNotNull()) &
    (F.col("tpep_dropoff_datetime").isNotNull()) &
    (F.col("PULocationID").isNotNull()) &
    (F.col("DOLocationID").isNotNull())
)

# Fill remaining nulls
df_clean = (df_clean
    .fillna({"congestion_surcharge": 0.0, "Airport_fee": 0.0, "RatecodeID": 1})
    .fillna({"store_and_fwd_flag": "N"})
)

clean_count = df_clean.count()
print(f"Records after cleaning:  {clean_count:,}")
print(f"Records removed:         {num_records - clean_count:,} ({((num_records - clean_count)/num_records)*100:.2f}%)")

STEP 1: DATA CLEANING
Records before cleaning: 41,169,720
Records after cleaning:  35,628,269


NameError: name 'num_records' is not defined

In [5]:
num_records = 41169720
removed = num_records - clean_count
print(f"Records removed: {removed:,} ({(removed/num_records)*100:.2f}%)")

Records removed: 5,541,451 (13.46%)


In [6]:
print("STEP 2: FEATURE ENGINEERING")
print("=" * 60)

df_features = df_clean.withColumns({
    # Temporal features
    "pickup_hour": F.hour("tpep_pickup_datetime"),
    "pickup_dayofweek": F.dayofweek("tpep_pickup_datetime"),
    "pickup_month": F.month("tpep_pickup_datetime"),
    "pickup_day": F.dayofmonth("tpep_pickup_datetime"),
    
    # Trip duration in minutes
    "trip_duration_min": (F.unix_timestamp("tpep_dropoff_datetime") - 
                          F.unix_timestamp("tpep_pickup_datetime")) / 60,
    
    # Speed (mph)
    "avg_speed_mph": F.when(
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) > 0,
        F.col("trip_distance") / ((F.unix_timestamp("tpep_dropoff_datetime") - 
         F.unix_timestamp("tpep_pickup_datetime")) / 3600)
    ).otherwise(0),
    
    # Is weekend
    "is_weekend": F.when(F.dayofweek("tpep_pickup_datetime").isin([1, 7]), 1).otherwise(0),
    
    # Time of day category
    "time_of_day": F.when(F.hour("tpep_pickup_datetime").between(6, 11), 0)
                    .when(F.hour("tpep_pickup_datetime").between(12, 16), 1)
                    .when(F.hour("tpep_pickup_datetime").between(17, 21), 2)
                    .otherwise(3),
    
    # Is rush hour
    "is_rush_hour": F.when(
        (F.hour("tpep_pickup_datetime").between(7, 9)) | 
        (F.hour("tpep_pickup_datetime").between(16, 19)), 1
    ).otherwise(0),
    
    # Fare per mile
    "fare_per_mile": F.when(F.col("trip_distance") > 0, 
                            F.col("fare_amount") / F.col("trip_distance")).otherwise(0),
    
    # Tip percentage
    "tip_percentage": F.when(F.col("fare_amount") > 0,
                             (F.col("tip_amount") / F.col("fare_amount")) * 100).otherwise(0),
    
    # Is airport trip
    "is_airport": F.when(F.col("RatecodeID") == 2, 1).otherwise(0)
})

# Filter invalid derived features
df_features = df_features.filter(
    (F.col("trip_duration_min") > 0) &
    (F.col("trip_duration_min") < 300) &
    (F.col("avg_speed_mph") < 100) &
    (F.col("avg_speed_mph") >= 0)
)

print(f"New features created: 12")
print(f"Total columns now: {len(df_features.columns)}")
print(f"Records after filtering: {df_features.count():,}")
print(f"\nNew Feature List:")
new_cols = ['pickup_hour','pickup_dayofweek','pickup_month','pickup_day',
            'trip_duration_min','avg_speed_mph','is_weekend','time_of_day',
            'is_rush_hour','fare_per_mile','tip_percentage','is_airport']
for c in new_cols:
    print(f"  - {c}")

STEP 2: FEATURE ENGINEERING
New features created: 12
Total columns now: 31
Records after filtering: 35,596,096

New Feature List:
  - pickup_hour
  - pickup_dayofweek
  - pickup_month
  - pickup_day
  - trip_duration_min
  - avg_speed_mph
  - is_weekend
  - time_of_day
  - is_rush_hour
  - fare_per_mile
  - tip_percentage
  - is_airport


In [7]:
print("STEP 3: ENCODE CATEGORICAL FEATURES")
print("=" * 60)

# Index store_and_fwd_flag (Y/N -> 0/1)
indexer = StringIndexer(inputCol="store_and_fwd_flag", outputCol="store_fwd_indexed")
df_features = indexer.fit(df_features).transform(df_features)

# Payment type is already numeric (1-5), keep as is
print("Categorical encoding complete!")
print(f"Total columns: {len(df_features.columns)}")

STEP 3: ENCODE CATEGORICAL FEATURES
Categorical encoding complete!
Total columns: 32


In [8]:
print("STEP 4: ASSEMBLE FEATURE VECTOR")
print("=" * 60)

feature_columns = [
    'trip_distance', 'passenger_count', 'PULocationID', 'DOLocationID',
    'payment_type', 'pickup_hour', 'pickup_dayofweek', 'pickup_month',
    'pickup_day', 'trip_duration_min', 'avg_speed_mph', 'is_weekend',
    'time_of_day', 'is_rush_hour', 'fare_per_mile', 'is_airport',
    'congestion_surcharge', 'Airport_fee', 'store_fwd_indexed'
]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features", handleInvalid="skip")
df_assembled = assembler.transform(df_features)

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_assembled)
df_final = scaler_model.transform(df_assembled)

print(f"Feature vector size: {len(feature_columns)}")
print(f"Features: {feature_columns}")
print(f"\nRecords in final dataset: {df_final.count():,}")

STEP 4: ASSEMBLE FEATURE VECTOR
Feature vector size: 19
Features: ['trip_distance', 'passenger_count', 'PULocationID', 'DOLocationID', 'payment_type', 'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'pickup_day', 'trip_duration_min', 'avg_speed_mph', 'is_weekend', 'time_of_day', 'is_rush_hour', 'fare_per_mile', 'is_airport', 'congestion_surcharge', 'Airport_fee', 'store_fwd_indexed']

Records in final dataset: 35,596,096


In [9]:
print("STEP 5: TRAIN/TEST SPLIT & SAVE")
print("=" * 60)

# 80/20 split
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

print(f"Training set: {train_df.count():,} records")
print(f"Test set:     {test_df.count():,} records")

# Save for Notebook 3
output_base = r"C:\Users\MaK Tech\Documents\GitHub\nyc-taxi-bigdata\data"

train_df.write.mode("overwrite").parquet(os.path.join(output_base, "train"))
test_df.write.mode("overwrite").parquet(os.path.join(output_base, "test"))

print(f"\nTrain data saved to: {output_base}\\train")
print(f"Test data saved to:  {output_base}\\test")

# Also save a sample CSV for Tableau
sample_for_tableau = df_features.sample(fraction=0.05, seed=42).toPandas()
tableau_path = os.path.join(output_base, "taxi_sample_tableau.csv")
sample_for_tableau.to_csv(tableau_path, index=False)
print(f"Tableau CSV saved:   {tableau_path} ({len(sample_for_tableau):,} rows)")

STEP 5: TRAIN/TEST SPLIT & SAVE
Training set: 28,475,545 records
Test set:     7,120,551 records


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "C:\Users\MaK Tech\anaconda3\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "C:\Users\MaK Tech\anaconda3\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\MaK Tech\anaconda3\Lib\socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
print("=" * 60)
print("NOTEBOOK 2 COMPLETE - FEATURE ENGINEERING")
print("=" * 60)
print(f"\nOriginal records:    41,169,720")
print(f"After cleaning:      35,628,269")
print(f"After feature eng:   {df_final.count():,}")
print(f"Features created:    12 new + 19 original = {len(df_final.columns)} total")
print(f"Feature vector size: {len(feature_columns)}")
print(f"Train/Test split:    80/20")
print(f"\nNext: Notebook 3 - Model Training")

spark.stop()
print("Spark session stopped.")

In [ ]:
print("=" * 60)
print("NOTEBOOK 2 COMPLETE - FEATURE ENGINEERING")
print("=" * 60)
print(f"\nOriginal records:    41,169,720")
print(f"After cleaning:      35,628,269")
print(f"Training set:        28,475,545")
print(f"Test set:            7,120,551")
print(f"Features created:    12 new features")
print(f"Feature vector size: 19")
print(f"Train/Test split:    80/20")
print(f"\nNext: Notebook 3 - Model Training")

spark.stop()
print("Spark session stopped.")

In [1]:
print("NOTEBOOK 2 COMPLETE - FEATURE ENGINEERING")
print("Original records:    41,169,720")
print("After cleaning:      35,628,269")
print("Training set:        28,475,545")
print("Test set:            7,120,551")
print("Features created:    12 new features")
print("Feature vector size: 19")
print("Train/Test split:    80/20")
print("Next: Notebook 3 - Model Training")

NOTEBOOK 2 COMPLETE - FEATURE ENGINEERING
Original records:    41,169,720
After cleaning:      35,628,269
Training set:        28,475,545
Test set:            7,120,551
Features created:    12 new features
Feature vector size: 19
Train/Test split:    80/20
Next: Notebook 3 - Model Training
